In [0]:
# Databricks Notebook — SalesLT Gold Layer
# ==========================================
# Sources : saleslt.silver (customer, product, sales_order_header, sales_order_detail)
#
# STAR SCHEMA
# ───────────
#   dim_customer        ← silver.customer
#   dim_product         ← silver.product  (no category join — not in silver)
#   dim_date            ← generated from order dates
#   fact_sales          ← silver.sales_order_detail + silver.sales_order_header
#
# AGGREGATIONS
#   agg_sales_by_customer   ← revenue, orders per customer
#   agg_sales_by_product    ← qty sold, revenue per product
#   agg_sales_by_month      ← monthly revenue trend
#
# CELL ORDER
# ──────────
#   CELL 0  — config
#   CELL 1  — create gold schema
#   CELL 2  — dim_customer
#   CELL 3  — dim_product
#   CELL 4  — dim_date
#   CELL 5  — fact_sales
#   CELL 6  — agg_sales_by_customer
#   CELL 7  — agg_sales_by_product
#   CELL 8  — agg_sales_by_month
#   CELL 9  — final summary
 
 
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — CONFIG
# ─────────────────────────────────────────────────────────────────────────────
 
SILVER = "saleslt.silver"
GOLD   = "saleslt.gold"
 
print("✓ Config loaded")
print(f"  Silver : {SILVER}")
print(f"  Gold   : {GOLD}")
print(f"  Sources: customer, product, sales_order_header, sales_order_detail")
 

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — DIM_CUSTOMER
# ─────────────────────────────────────────────────────────────────────────────
# Grain   : one row per customer
# Source  : silver.customer
# Adds    : full_name, customer_segment (B2B / B2C)
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_customer
    USING DELTA AS
 
    SELECT
        CustomerID                                AS customer_key,
        FirstName                                 AS first_name,
        LastName                                  AS last_name,
        CONCAT(FirstName, ' ', LastName)          AS full_name,
        EmailAddress                              AS email,
        Phone                                     AS phone,
        CompanyName                               AS company_name,
        CASE
            WHEN CompanyName IS NOT NULL THEN 'B2B'
            ELSE 'B2C'
        END                                       AS customer_segment
    FROM {SILVER}.customer
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_customer").collect()[0]["n"]
print(f"✓ dim_customer       — {n:,} rows")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — DIM_PRODUCT
# ─────────────────────────────────────────────────────────────────────────────
# Grain   : one row per product
# Source  : silver.product only (product_category not in silver)
# Adds    : gross_margin, price_range, product_status
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_product
    USING DELTA AS
 
    SELECT
        ProductID                                 AS product_key,
        ProductNumber                             AS product_number,
        Name                                      AS product_name,
        Color                                     AS color,
        Size                                      AS size,
        Weight                                    AS weight,
        StandardCost                              AS standard_cost,
        ListPrice                                 AS list_price,
        ROUND(ListPrice - StandardCost, 2)        AS gross_margin,
        CASE
            WHEN ListPrice = 0     THEN 'Free'
            WHEN ListPrice < 100   THEN 'Budget'
            WHEN ListPrice < 500   THEN 'Mid-Range'
            WHEN ListPrice < 2000  THEN 'Premium'
            ELSE 'Luxury'
        END                                       AS price_range,
        SellStartDate                             AS sell_start_date,
        SellEndDate                               AS sell_end_date,
        CASE
            WHEN SellEndDate IS NULL THEN 'Active'
            ELSE 'Discontinued'
        END                                       AS product_status
    FROM {SILVER}.product
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_product").collect()[0]["n"]
print(f"✓ dim_product        — {n:,} rows")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — DIM_DATE
# ─────────────────────────────────────────────────────────────────────────────
# Grain   : one row per calendar date
# Source  : generated from the min/max range of OrderDate in silver
# Columns : standard date attributes for time-intelligence in BI tools
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_date
    USING DELTA AS
 
    WITH date_spine AS (
        SELECT EXPLODE(
            SEQUENCE(
                MIN(TO_DATE(OrderDate)),
                MAX(TO_DATE(OrderDate)),
                INTERVAL 1 DAY
            )
        ) AS date
        FROM {SILVER}.sales_order_header
    )
    SELECT
        date                                      AS date_key,
        DATE_FORMAT(date, 'yyyyMMdd')             AS date_id,
        YEAR(date)                                AS year,
        QUARTER(date)                             AS quarter,
        MONTH(date)                               AS month_number,
        DATE_FORMAT(date, 'MMMM')                 AS month_name,
        DATE_FORMAT(date, 'MMM yyyy')             AS month_year,
        WEEKOFYEAR(date)                          AS week_of_year,
        DAYOFMONTH(date)                          AS day_of_month,
        DATE_FORMAT(date, 'EEEE')                 AS day_name,
        CASE
            WHEN DAYOFWEEK(date) IN (1,7) THEN TRUE
            ELSE FALSE
        END                                       AS is_weekend,
        CONCAT(YEAR(date), '-Q', QUARTER(date))   AS quarter_label
    FROM date_spine
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_date").collect()[0]["n"]
print(f"✓ dim_date           — {n:,} rows")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — FACT_SALES
# ─────────────────────────────────────────────────────────────────────────────
# Grain   : one row per order line (SalesOrderDetail)
# Source  : silver.sales_order_detail + silver.sales_order_header
# FK keys : customer_key → dim_customer
#           product_key  → dim_product
#           date_key     → dim_date
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_sales
    USING DELTA AS
 
    SELECT
        -- Foreign keys to dimensions
        h.CustomerID                                      AS customer_key,
        d.ProductID                                       AS product_key,
        TO_DATE(h.OrderDate)                              AS date_key,
 
        -- Degenerate dimensions (order identifiers kept on fact)
        h.SalesOrderID                                    AS sales_order_id,
        d.SalesOrderDetailID                              AS sales_order_detail_id,
 
        -- Order dates
        h.OrderDate                                       AS order_date,
        h.DueDate                                         AS due_date,
        h.ShipDate                                        AS ship_date,
 
        -- Line-level measures
        d.OrderQty                                        AS order_qty,
        d.UnitPrice                                       AS unit_price,
        d.UnitPriceDiscount                               AS discount_rate,
        ROUND(d.UnitPrice * d.UnitPriceDiscount
              * d.OrderQty, 2)                            AS discount_amount,
        d.LineTotal                                       AS line_total,
 
        -- Order-level measures (from header)
        h.SubTotal                                        AS order_sub_total,
        h.TaxAmt                                          AS order_tax,
        h.Freight                                         AS order_freight,
        h.TotalDue                                        AS order_total_due,
 
        -- Derived measures
        ROUND(d.LineTotal
              - (d.UnitPrice * d.UnitPriceDiscount
              * d.OrderQty), 2)                           AS revenue_after_discount,
        CASE
            WHEN h.ShipDate IS NOT NULL THEN
                DATEDIFF(TO_DATE(h.ShipDate), TO_DATE(h.OrderDate))
            ELSE NULL
        END                                               AS days_to_ship
 
    FROM      {SILVER}.sales_order_detail  d
    JOIN      {SILVER}.sales_order_header  h
           ON d.SalesOrderID = h.SalesOrderID
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.fact_sales").collect()[0]["n"]
print(f"✓ fact_sales         — {n:,} rows")
 

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — AGG : SALES BY CUSTOMER
# ─────────────────────────────────────────────────────────────────────────────
# Business question : Who are our best customers by revenue?
# Grain   : one row per customer
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.agg_sales_by_customer
    USING DELTA AS
 
    SELECT
        f.customer_key,
        c.full_name,
        c.email,
        c.customer_segment,
        COUNT(DISTINCT f.sales_order_id)              AS total_orders,
        SUM(f.order_qty)                              AS total_qty_purchased,
        ROUND(SUM(f.line_total), 2)                   AS total_revenue,
        ROUND(AVG(f.line_total), 2)                   AS avg_line_value,
        ROUND(SUM(f.line_total)
              / COUNT(DISTINCT f.sales_order_id), 2)  AS avg_order_value,
        MIN(f.order_date)                             AS first_order_date,
        MAX(f.order_date)                             AS last_order_date,
        DATEDIFF(
            MAX(TO_DATE(f.order_date)),
            MIN(TO_DATE(f.order_date))
        )                                             AS customer_lifespan_days
    FROM      {GOLD}.fact_sales    f
    JOIN      {GOLD}.dim_customer  c
           ON f.customer_key = c.customer_key
    GROUP BY
        f.customer_key, c.full_name,
        c.email,        c.customer_segment
    ORDER BY total_revenue DESC
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.agg_sales_by_customer").collect()[0]["n"]
print(f"✓ agg_sales_by_customer  — {n:,} rows")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — AGG : SALES BY PRODUCT
# ─────────────────────────────────────────────────────────────────────────────
# Business question : Which products drive the most revenue and volume?
# Grain   : one row per product
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.agg_sales_by_product
    USING DELTA AS
 
    SELECT
        f.product_key,
        p.product_name,
        p.price_range,
        p.product_status,
        COUNT(DISTINCT f.sales_order_id)              AS total_orders,
        SUM(f.order_qty)                              AS total_qty_sold,
        ROUND(SUM(f.line_total), 2)                   AS total_revenue,
        ROUND(AVG(f.unit_price), 2)                   AS avg_selling_price,
        ROUND(AVG(f.discount_rate) * 100, 2)          AS avg_discount_pct,
        ROUND(SUM(f.line_total)
              / NULLIF(SUM(f.order_qty), 0), 2)       AS revenue_per_unit
    FROM      {GOLD}.fact_sales   f
    JOIN      {GOLD}.dim_product  p
           ON f.product_key = p.product_key
    GROUP BY
        f.product_key, p.product_name,
        p.price_range, p.product_status
    ORDER BY total_revenue DESC
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.agg_sales_by_product").collect()[0]["n"]
print(f"✓ agg_sales_by_product   — {n:,} rows")
 

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — AGG : SALES BY MONTH
# ─────────────────────────────────────────────────────────────────────────────
# Business question : What is the monthly revenue trend?
# Grain   : one row per year-month
 
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.agg_sales_by_month
    USING DELTA AS
 
    SELECT
        d.year,
        d.month_number,
        d.month_name,
        d.month_year,
        d.quarter_label,
        COUNT(DISTINCT f.sales_order_id)              AS total_orders,
        COUNT(f.sales_order_detail_id)                AS total_order_lines,
        SUM(f.order_qty)                              AS total_qty_sold,
        ROUND(SUM(f.line_total), 2)                   AS total_revenue,
        ROUND(AVG(f.line_total), 2)                   AS avg_line_value,
        ROUND(AVG(f.days_to_ship), 1)                 AS avg_days_to_ship
    FROM      {GOLD}.fact_sales  f
    JOIN      {GOLD}.dim_date    d
           ON f.date_key = d.date_key
    GROUP BY
        d.year, d.month_number, d.month_name,
        d.month_year, d.quarter_label
    ORDER BY
        d.year, d.month_number
""")
 
n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.agg_sales_by_month").collect()[0]["n"]
print(f"✓ agg_sales_by_month     — {n:,} rows")
 
 

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "=" * 58)
print("  GOLD LAYER SUMMARY")
print("=" * 58)
 
gold_tables = {
    "dim_customer":           "DIM ",
    "dim_product":            "DIM ",
    "dim_date":               "DIM ",
    "fact_sales":             "FACT",
    "agg_sales_by_customer":  "AGG ",
    "agg_sales_by_product":   "AGG ",
    "agg_sales_by_month":     "AGG ",
}
 
for table, kind in gold_tables.items():
    n = spark.sql(
        f"SELECT COUNT(*) AS n FROM {GOLD}.{table}"
    ).collect()[0]["n"]
    print(f"  {kind}  {table:35s}  {n:>8,} rows")
 
print("=" * 58)
print(f"\n  Star schema ready for BI / reporting tools")
print(f"  Connect Power BI to : {GOLD}")